# Intelligent Agents: Reflex-Based Agents for GridHunt

Student Name: [Add your name]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [your initials]

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: Undergrads 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and the hunter and the monster move on this grid. A hunter catches the monster if she/he moves on the same square the monster currently occupies. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunter are randomly placed in the arena environment. The hunter and the monster
    can move around, but cannot leave the arena.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena but only gets the monster location when it stops and listens.

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [371]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_agent_function, 
                      monster_agent_function, 
                      n = 30, max_steps = 100, 
                      visualize = False, animation = False):
    """
    Simulate an arena where a hunter agent tries to catch a monster agent.

    Parameters:
    hunter_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_pos = np.random.randint(0, n-1, size=2)
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step+1}: Hunter is at '{hunter_pos}'; Monster is at '{monster_pos}'")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_pos[0], hunter_pos[1]] = 'H'
        if np.array_equal(hunter_pos, monster_pos):
            arena[monster_pos[0], monster_pos[1]] = '*'
        print("\n".join("".join(row) for row in arena))

    # run the environment
    hunter_action = None  # Initialize hunter_action to None for the first step
    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_pos, monster_pos):
            if visualize:
                print(f"Hunter caught the monster in {step} steps!")
            return step
        
        # Get next move from monster agent function
        monster_action = monster_agent_function(hunter_pos, monster_pos)
        
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function. The monster's position is only provided 
        # if the hunter chose 'listen' as the previous action.
        if hunter_action == 'listen':
            monster_pos_instrument = monster_pos
        else: 
            monster_pos_instrument = None
        
        hunter_action = hunter_agent_function(hunter_pos, monster_pos_instrument)
        
        if hunter_action == 'listen':
            pass
        elif hunter_action == 'teleport':
            hunter_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_pos = move(hunter_action, hunter_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter chose action '{hunter_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Hunter failed to catch the monster in {max_steps} steps.")
    
    return np.nan

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [372]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(hunter_pos, monster_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [373]:
actions = ["north", "east", "west", "south", "teleport", "listen"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [374]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [375]:
simple_randomized_hunter_agent_function([0,0], None)

np.str_('south')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [376]:
arena_environment(simple_randomized_hunter_agent_function, 
                  monster_agent_function_simple, 
                  n=5, max_steps=10, visualize=True, animation=False)

Step 1: Hunter is at '[0 3]'; Monster is at '[3 1]'
...H.
.....
.....
.M...
.....
Hunter chose action 'west'
Monster chose action 'west'


Step 2: Hunter is at '[0 2]'; Monster is at '[3 0]'
..H..
.....
.....
M....
.....
Hunter chose action 'east'
Monster chose action 'east'


Step 3: Hunter is at '[0 3]'; Monster is at '[3 1]'
...H.
.....
.....
.M...
.....
Hunter chose action 'east'
Monster chose action 'west'


Step 4: Hunter is at '[0 4]'; Monster is at '[3 0]'
....H
.....
.....
M....
.....
Hunter chose action 'north'
Monster chose action 'south'


Step 5: Hunter is at '[0 4]'; Monster is at '[4 0]'
....H
.....
.....
.....
M....
Hunter chose action 'teleport'
Monster chose action 'stay'


Step 6: Hunter is at '[0 3]'; Monster is at '[4 0]'
...H.
.....
.....
.....
M....
Hunter chose action 'listen'
Monster chose action 'west'


Step 7: Hunter is at '[0 3]'; Monster is at '[4 0]'
...H.
.....
.....
.....
M....
Hunter chose action 'listen'
Monster chose action 'stay'


Step 8: Hunter is

nan

**Note for VS Code:** View as scrollable element to see the complete output.

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $10 \times 10$ arena.  

In [377]:
steps = [arena_environment(simple_randomized_hunter_agent_function, monster_agent_function_simple, n=10, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[27, nan, 43, 12, nan, 11, nan, nan, 15, 7, 23, nan, 13, nan, nan, nan, 18, 37, nan, 19, 1, nan, nan, 52, nan, nan, 14, 36, nan, 21, 17, 4, 32, nan, nan, nan, 74, nan, nan, 3, nan, nan, 35, nan, nan, 30, 53, nan, 51, nan, nan, 27, nan, 48, nan, nan, nan, 67, nan, 19, 92, nan, nan, nan, 90, nan, 57, 44, 10, nan, nan, 13, nan, 43, 50, nan, nan, 59, nan, nan, 75, nan, nan, nan, 1, 40, 79, 33, nan, nan, nan, nan, nan, nan, nan, nan, 30, 14, 63, nan]


We just got a list with the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results. `nan` means that the hunter was not able to catch the monster. How many times did that happen? To answer how well the hunter did when it caught the monster, we just average the numbers that are not `nan`. 

In [378]:
print (f"Hunter failed {np.sum(np.isnan(steps))} times.")
print (f"Average steps to catch the monster over {np.sum(~np.isnan(steps))} successful runs: {round(np.nanmean(steps), 2)}")

Hunter failed 54 times.
Average steps to catch the monster over 46 successful runs: 34.83


This in not a very good agent. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and make it into a model-based reflex agent. It should use listen to get the monster's location and then move towards the monster. To do that it needs to remember the monster's location. This memory makes it model-based. Here is HOWTO: How to Store State Information in the Agent.

In [379]:
# here goes your agent implementation
class ModelBasedHunter:
    def __init__(self):
        # Trí nhớ của thợ săn về vị trí quái vật
        self.known_monster_pos = None

    def __call__(self, hunter_pos, monster_pos_instrument):
        # 1. Cập nhật trí nhớ nếu vừa "listen" ở bước trước
        if monster_pos_instrument is not None:
            self.known_monster_pos = monster_pos_instrument.copy()
            
        # 2. Nếu chưa biết quái vật ở đâu, hoặc đã đi đến vị trí cũ mà quái vật đã bỏ đi -> phải listen lại
        if self.known_monster_pos is None or np.array_equal(hunter_pos, self.known_monster_pos):
            return 'listen'
            
        # 3. Phản xạ di chuyển dựa trên vị trí đã nhớ (Rút ngắn khoảng cách)
        hx, hy = hunter_pos
        mx, my = self.known_monster_pos
        
        # Trong mảng Numpy: north (hàng giảm), south (hàng tăng), west (cột giảm), east (cột tăng)
        if hx > mx:
            return 'north'
        elif hx < mx:
            return 'south'
        elif hy > my:
            return 'west'
        elif hy < my:
            return 'east'
            
        return 'listen' # Fallback an toàn

# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [380]:
# Your experimentation code goes here
def run_experiment(arena_size, runs=100, max_steps=1000):
    steps_record = []
    
    for _ in range(runs):
        # Khởi tạo lại thợ săn mới (xóa trí nhớ cũ) cho mỗi ván
        hunter_agent = ModelBasedHunter()
        
        # Chạy môi trường
        steps = arena_environment(
            hunter_agent, 
            monster_agent_function_simple, 
            n=arena_size, 
            max_steps=max_steps, 
            visualize=False
        )
        steps_record.append(steps)
        
    # Xử lý số liệu
    steps_array = np.array(steps_record)
    success_count = np.sum(~np.isnan(steps_array))
    success_rate = (success_count / runs) * 100
    avg_steps = np.nanmean(steps_array)
    
    print(f"--- Đấu trường {arena_size}x{arena_size} ---")
    print(f"Tỷ lệ bắt thành công: {success_rate}%")
    print(f"Số bước trung bình: {avg_steps:.2f} bước")
    print(f"Ván chậm nhất: {np.nanmax(steps_array):.0f} bước\n")

# Thực thi thử nghiệm với 2 kích thước bản đồ
np.random.seed(1)
run_experiment(arena_size=10, runs=1000, max_steps=1000)
np.random.seed(1)
run_experiment(arena_size=30, runs=1000, max_steps=1000)

--- Đấu trường 10x10 ---
Tỷ lệ bắt thành công: 100.0%
Số bước trung bình: 9.42 bước
Ván chậm nhất: 32 bước

--- Đấu trường 30x30 ---
Tỷ lệ bắt thành công: 100.0%
Số bước trung bình: 25.12 bước
Ván chậm nhất: 74 bước



## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

**1. Chiến lược chọn hành động và lý do hiệu quả**

Thợ săn là tác tử phản xạ dựa trên mô hình (model-based reflex agent) với một biến trạng thái `known_monster_pos` lưu vị trí quái vật lần cuối nghe được. Mỗi lượt nó áp dụng ba luật theo thứ tự:
- Nếu vừa `listen` ở lượt trước thì cập nhật bộ nhớ bằng vị trí mới.
- Nếu chưa biết vị trí quái vật, hoặc đã đến đúng ô đã nhớ mà chưa bắt được (quái vật đã đi chỗ khác), thì `listen`.
- Ngược lại, đi một bước làm giảm khoảng cách Manhattan đến vị trí đã nhớ (chỉnh hàng trước, rồi cột).

Chiến lược này hiệu quả vì mỗi bước di chuyển đều làm khoảng cách giảm đúng 1, nên thợ săn không đi lang thang như tác tử ngẫu nhiên. `listen` cũng tốn một bước nên chỉ dùng khi cần (đầu ván hoặc khi đến nơi mà không thấy quái vật). Quái vật đứng yên 50% thời gian và đi ngẫu nhiên đối xứng, nên vị trí đã nhớ vẫn gần đúng trong vài chục bước. Kết quả: bắt được quái vật trong 100% số ván, trung bình khoảng 10 bước ở 10×10 và 25 bước ở 30×30, trong khi tác tử ngẫu nhiên chỉ thành công khoảng 50% (10×10) và 8% (30×30) trong 100 bước.

**2. Teleportation**

Thợ săn cuối cùng **không** dùng teleport. Teleport đưa thợ săn đến một ô ngẫu nhiên, tức là đặt lại khoảng cách đến quái vật về giá trị kỳ vọng khoảng 2n/3 và tốn thêm một bước. Vì thợ săn đã biết hướng đi đến quái vật, mỗi bước đi bộ đều chắc chắn tiến gần hơn, còn teleport thì không. Teleport chỉ có thể có lợi khi thợ săn đang ở rất xa vị trí đã nhớ (nhất là ở đấu trường lớn); khi đã ở gần thì nó gần như chắc chắn làm thợ săn xa hơn. Vì vậy em chọn bản đơn giản, ổn định và không cần teleport.

**3. Ảnh hưởng của kích thước đấu trường**

Với thợ săn có bộ nhớ, số bước tăng gần tuyến tính theo n vì nó chủ yếu phải đi hết khoảng cách ban đầu giữa hai vị trí ngẫu nhiên (kỳ vọng khoảng 2n/3): từ 10×10 lên 30×30 (diện tích tăng 9 lần) số bước chỉ tăng khoảng 2,5–2,7 lần. Tỷ lệ thành công vẫn là 100%. Ngược lại, tác tử ngẫu nhiên suy giảm rất mạnh khi lưới lớn vì diện tích cần tìm tăng theo n² mà nó không có hướng đi: tỷ lệ thành công trong 100 bước rơi từ khoảng 50% xuống khoảng 8%.> Your discussion goes here

The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

Quái vật nhận được vị trí của thợ săn ở mọi lượt, nên hàm tác tử của nó có thể được cải thiện như sau:
- **Chạy trốn có định hướng:** thay vì đi ngẫu nhiên và đứng yên 50% thời gian, chọn hướng làm tăng khoảng cách Manhattan đến thợ săn. Khi đó vị trí thợ săn đã nghe được sẽ nhanh chóng lỗi thời và nó phải đuổi theo một mục tiêu đang di chuyển.
- **Tránh mép và góc lưới:** khi chạy trốn nên ưu tiên hướng còn nhiều chỗ để đi, nếu không sẽ bị dồn vào góc.
- **Chỉ chạy khi cần:** đứng yên khi thợ săn còn xa (để đỡ lộ vị trí mới sau khi thợ săn `listen`) và chạy khi thợ săn đến gần.
- **Thêm yếu tố khó đoán:** đôi khi chọn hướng khác thay vì luôn hướng ra xa, để thợ săn không thể tính trước đường đi.
- **Đề phòng teleport:** thợ săn có thể nhảy đến ô ngẫu nhiên, nên quái vật không nên chỉ dựa vào khoảng cách hiện tại mà cũng nên tránh các vị trí dễ bị dồn ép.

Đây là các đề xuất chưa được thực nghiệm; để kiểm chứng có thể viết một `monster_agent_function` mới rồi chạy lại thí nghiệm với thợ săn hiện tại.> Your discussion goes here

# More Work (Optional)

Here are some ideas:

* Implement a better monster agent function and perform experiments
* Change the environment so multiple hunters can hunt the monster.
* Give the hunter a new action that shoots an arrow in a specific direction. 
  The arrow can go a maximum of 5 squares. If the hunter hits the monster, then it wins.